# Classify with a neural network

**What it does.** Train a convolutional network on cropped single-object images and score every object in the screen.

**When to use it.** When the phenotype is visible but hard to write down as a rule — the classifier learns it from labelled examples.

**What you get.** A trained model, a held-out performance report, and per-object scores merged back into the measurement database.

---

> Every path below is a placeholder. Point `src` at your own data before running.
> Nothing in this notebook writes outside the folder you give it.

## 1. Check the install

If this cell fails, the rest cannot work. It reports the version and whether a GPU is visible — segmentation and training are usable on CPU but slow.

In [ ]:
import spacr
from spacr.version import version_str

print(version_str)

## 2. The function this notebook runs

`spacr.deep_spacr.deep_spacr`

```
deep_spacr(settings=None)
```

Run the full spacr deep-learning pipeline: build dataset, train, apply model, merge predictions into the measurements DB.

In [ ]:
from spacr.deep_spacr import deep_spacr

## 3. Settings and API reference

Read the descriptions here, then edit only the values in the next cell. Defaults and descriptions are generated from the installed spaCR version, so the notebook stays aligned with the API.

### [`spacr.deep_spacr.deep_spacr`](https://einarolafsson.github.io/spacr/api/spacr/deep_spacr/index.html#spacr.deep_spacr.deep_spacr)

- **`amsgrad`** — (bool) - Use the AMSGrad variant of Adam/AdamW, which keeps a running maximum of past squared gradients instead of their decaying average so the effective step size never grows back. Enable when training loss oscillates or stops converging with plain Adam; it costs a little speed and memory. Only honoured by optimizer_type 'adam' and 'adamw' - ignored by sgd, rmsprop, nadam, radam and adagrad. Default True.
- **`annotation_column`** — (str) - Integer column of the png_list table holding manual class calls. The Annotate app adds it with ALTER TABLE if missing and writes labels into it. It is the ground truth when dataset_mode is 'annotation', and the fallback when annotation_columns is unset. Setting it while leaving dataset_mode unset also SELECTS annotation mode, which is how an old settings file keeps working. Default None.
- **`apply_model_to_dataset`** — (bool) - After training (or straight away when reusing a saved model_path), pack the object PNGs into a tar, run inference over it, copy the n_top_examples most confident images per class into top_examples/, and merge the per-object scores back into measurements.db. Turn it off to only train and evaluate a model without scoring the screen. Default True.
- **`augment`** — (bool) - Expand the training split 8-fold by adding all four 90-degree rotations of each crop plus their horizontal mirrors; the validation and test splits are never augmented. Turn it on when you have few annotated objects and validation accuracy lags training accuracy. The expanded set is materialised in RAM, so expect roughly 8x the memory and 8x the epoch time. Default False.
- **`balance_to_smallest`** — (bool) - Downsample every generated training class to the size of the smallest class before writing train/test folders. This removes the dataset prior but discards majority examples; disable it and use class_balance during training to retain all images. Default True.
- **`batch_size`** — (int) - How many images are held and processed together in one pass: field stacks during normalization and Cellpose segmentation, crops per step during classifier training and activation maps. Raising it speeds runs up but increases RAM/VRAM roughly linearly; lower it on out-of-memory errors. Defaults: 50 for mask generation, 64 for training.
- **`channel_of_interest`** — (int) - Index of the fluorescence channel the downstream analysis focuses on. It decides which channel's features survive filtering (other channels' features are dropped), defines recruitment = pathogen_channel_N_mean_intensity / cytoplasm_channel_N_mean_intensity, and is written into the ML result paths. Set it to the channel carrying your phenotype readout. Valid 0-3; default 3 in the ML/recruitment steps, 1-2 elsewhere.
- **`class_balance`** — (str) - How skew between training classes is corrected. 'none' changes nothing but still prints the per-class counts, the ratio and a recommendation, so the skew is never invisible. 'weighted_sampler' draws every class about equally often (1/n). 'sqrt_weighted_sampler' uses 1/sqrt(n), a partial correction that avoids oversampling a tiny class so hard the model memorises its few crops. 'weighted_loss' leaves sampling alone and weights the loss. Reach for one of the latter three when the printed ratio is worse than about 3:1. Default 'none'.
- **`class_folder_names`** — (list of str) - Ordered training folder names. Each must exactly match a subfolder under src/train and src/test; a name's position in this list becomes its integer label, and the list length sets the width of the classifier head. Training raises a FileNotFoundError listing missing vs available folders if a name has no folder. Generate Training Dataset overwrites this with the names it actually wrote to disk. This is where the crops ARE; 'classes' is what they MEAN. Default ['nc','pc'].
- **`class_metadata`** — (list of lists) - One inner list per training class, holding the metadata values that select that class's objects, for example [['c1'],['c2']] for a two-class run keyed on column. Order fixes the class indices the model learns, so reordering the inner lists relabels the whole training set. Values that occur in no row make the generator select nothing and stop. Default [['c1'], ['c2']].
- **`classes`** — (dict) - What each class MEANS: class name -&gt; {column, value}, so 'pc' might be {'column': 'columnID', 'value': 'c3'}. Set it in the Classes editor: pick a column and every distinct value becomes a row you name, across several columns if you need. One row may instead be a random complement - everything no other rule claimed, sampled to match the largest class. This says which OBJECTS belong to a class; the training subfolder names are class_folder_names. A pre-split settings file holds a plain list and is translated on read. Default {}.
- **`classifier_evaluation`** — (bool) - Build the Classifier Evaluation workbench bundle from out-of-fold predictions after Classify (CV): sample-level predictions, confusion matrices, reliability curves, calibrated probabilities, per-plate metrics, leakage reports and a manifest. It requires cross_validation_folds &gt;= 2; a single train/validation split cannot produce unbiased out-of-fold diagnostics. Default True. API: spacr.classifier_evaluation.evaluate_predictions.
- **`coordinate_columns`** — (list) - On-demand crops from a DATABASE instead of masks: the columns holding each object's position, e.g. ['centroid_x', 'centroid_y']. Only bounding-box crops are possible this way, because a coordinate has no outline. None uses the merged masks, which is the default and the better source. Default None.
- **`crop_shape`** — (str) - 'bounding_box' cuts the smallest rectangle containing the object; 'object' masks everything outside it away. Database-sourced crops can only be bounding boxes. Default 'bounding_box'.
- **`crop_source`** — (str) - Select where image crops come from. Viewers use 'png' (LOAD IMAGES) for exported crops in data/ or 'merged' (STREAM IMAGES) to cut from merged/*.npy using the measurements database; spaCR reports any fallback. Training uses 'pre_generated' for existing crops, 'on_demand' to cut during training, or 'generate' to write a crop set first. Controls that do not apply to the selected source are disabled. Default 'png' in viewers and 'pre_generated' in training.
- **`cross_validation_enabled`** — (bool) - Enable k-fold validation for Classify. If cross_validation_folds is 0 or 1, enabling this uses 5 folds. Use cv_group_by='plate' to hold out whole plates, or 'well'/'field' for within-plate validation without leaking related crops between training and validation. Default False.
- **`cross_validation_folds`** — (int) - Number of k-fold splits the vision classifier is trained with in place of the single val_split hold-out. 0 (the default) or 1 keeps today's one random split; 2 or more trains a fresh model per fold, scores each on the fold it never saw, and reports the mean together with the fold-to-fold standard deviation and range - which is the only way to see whether one lucky split was flattering the model. Costs roughly k times the training time. Distinct from 'cross_validation', which is the regression pipeline's own toggle.
- **`custom_model`** — (str) - Path to a saved Cellpose model, loaded as pretrained_model by the mask-finetune tool. When set, model_type is passed as None and diameter as diam_mean (which Cellpose 4.x ignores with a warning), but model_name is STILL read: it selects the channel pair sent to model.eval. So a custom model with the wrong model_name segments the wrong channels. Default None.
- **`custom_model_path`** — (str) - Path to a trained classifier artifact whose model weights initialize a new fine-tuning run. The optimizer and epoch start fresh. Leave empty to initialize from ImageNet or random weights according to init_weights. Default ''.
- **`cv_group_by`** — (str) - Train/test independence: 'cell', 'field', 'well' (default), or 'plate'. Cell can place sibling crops from one well on both sides; field narrows but does not close that leak; well matches the usual experimental assignment unit; plate holds out a complete batch. Whole groups make the requested fraction approximate, so runs report held-out groups and cells. Legacy 'none'/'off' alias 'cell'. Crop identities come from spaCR's plate_well_field_object.png names; unverifiable grouped designs are refused rather than silently randomized.
- **`dataset`** — (str) - Path to the .tar archive of single-object PNG crops produced by generate_dataset, which the activation-map step opens with TarImageDataset. The plate folder is inferred two levels above it and CAM outputs are written next to it under &lt;tar_name&gt;/&lt;cam_type&gt;/. Must be a full path, not just a file name. Default ''.
- **`dataset_mode`** — (str) - How training classes are defined: 'metadata' splits crops by well metadata (class_metadata or metadata_rules), 'annotation' by the values in one or more annotation columns of png_list, 'measurement' by threshold rules on measured features (measurement_rules). Any other value aborts and returns no dataset. Default 'metadata'.
- **`dropout_rate`** — (float) - Dropout probability (0-1) written into every existing Dropout layer of the backbone and applied to a Dropout inserted before the final linear classifier; 0 or None removes dropout entirely. Raise it (0.2-0.5) when training accuracy runs well ahead of validation accuracy; lower it when the model underfits and training loss stalls high. Default 0.1.
- **`early_stopping_patience`** — (int) - Stop training after this many consecutive epochs in which validation accuracy fails to beat the best value so far; the best checkpoint is still kept. 0 (default) disables it and always runs the full 'epochs' budget. Set 10-20 on long runs to cut wasted epochs once the model plateaus.
- **`epochs`** — (int) - Number of full passes over the training set. It also sets the learning-rate schedule horizon - cosine anneals over exactly this many epochs and step_lr drops every epochs/5 - so changing it rescales the schedule. A checkpoint is always written on the final epoch and every 100th. Raise it for small datasets and use early_stopping_patience to cut runs short. Default 100.
- **`evaluation_bins`** — (int) - Number of equal-width probability bins in reliability curves and expected calibration error. Values around 10 balance resolution against noise; use fewer bins for small validation sets and more only when every class has many hundreds of out-of-fold samples. Minimum 2, default 10. API: spacr.classifier_evaluation.calibration_table.
- **`evaluation_calibration`** — (str) - Probability calibration written to the evaluation bundle. 'temperature' cross-fits one scalar temperature per held-out fold using all other out-of-fold predictions, so a sample never fits its own calibrator; 'none' retains raw softmax probabilities. Calibration changes reported probabilities, not the saved model weights. Default 'temperature'. API: spacr.classifier_evaluation.cross_calibrate_probabilities.
- **`evaluation_fail_on_leakage`** — (bool) - Stop Classify (CV) before fitting a fold when the same object, augmentation family, or protected cv_group_by identity appears in both train and validation. False records the problem and continues, which is useful only for diagnosing a legacy dataset because its performance estimate remains invalid. Default True. API: spacr.classifier_evaluation.audit_split_leakage.
- **`experiment`** — (str) - Free-text run label. Its real effect is naming the exported PNG dataset tar as &lt;YYMMDD&gt;_&lt;experiment&gt;.tar (a random-numbered variant is used if that name already exists), so give each screen a distinct value to avoid confusing dataset tars. It is also passed to the measurement-database writer but not stored there. Defaults vary by pipeline: 'exp', 'exp.' or 'experiment_1'.
- **`extract_channels`** — (list) - On-demand crops: which planes of merged/*.npy are INTENSITY channels, in the order they become image channels. Default [0, 1, 2].
- **`file_metadata`** — (str, list or None) - Substring filter applied to png_path when pulling crops from the database: only paths containing it are included, and a list matches any one of its entries (OR, not AND). Use it to restrict a dataset to one plate, well or object type, e.g. 'plate1_' or 'cell_png'. None takes every crop. Default None.
- **`file_type`** — (str) - Substring that selects which object crops go into the dataset - only png_list rows whose PNG path contains it are used, e.g. 'cell_png', 'nucleus_png', 'pathogen_png', 'cytoplasm_png' or 'organelle_png'. In the GUI this one field writes both file_type and png_type, and only png_type is read downstream, so the two always hold the same value. Default 'cell_png'.
- **`focal_alpha`** — (float) - Class-balancing weight for focal loss (read only when loss_type resolves to focal). In the single-logit binary path it scales positives by alpha and negatives by 1-alpha, so raise it toward 1 to emphasise a rare positive class; with two or more output classes a plain float scales the whole loss uniformly. Default None (no alpha weighting).
- **`focal_gamma`** — (float) - Focusing exponent in the focal-loss weight (1 - p_t)^gamma, applied only when loss_type is focal. 0 reduces it to plain cross-entropy; raising it (typically 1-5) down-weights crops the model already classifies well and pushes gradient onto hard ones. Default 2.0. Increase when one class dominates and training stalls on easy examples.
- **`generate_full_dataset`** — (bool) - Build the full unlabelled inference dataset tar from every selected plate independently of training or model application. Apply model to dataset also creates it automatically when needed. API: spacr.io.generate_dataset. Default False.
- **`generate_training_dataset`** — (bool) - Rebuild the train/ and test/ PNG folders from the object crops before training, using the dataset_mode rules (annotation_column labels, metadata rules or measurement rules) and splitting off test_split of the images. Turn it off to reuse an existing split; it is only consulted when train or test is True, and a failed build aborts training. Default True.
- **`gradient_accumulation`** — (bool) - Sum gradients over several batches before each optimizer step instead of stepping on every batch, giving an effective batch size of batch_size x gradient_accumulation_steps without extra GPU memory. Enable when you had to shrink batch_size to fit in VRAM and training is noisy. Leftover gradients are flushed at the end of each epoch. Default True.
- **`gradient_accumulation_steps`** — (int) - How many batches are summed per optimizer step when gradient_accumulation is on; the loss is divided by this value so gradient magnitude stays comparable. Effective batch size = batch_size x this. Raise it (4-16) to emulate a larger batch on limited VRAM, at the cost of fewer weight updates per epoch. Ignored when gradient_accumulation is False. Default 4.
- **`image_size`** — (int) - Side length in pixels of the centre crop taken from each object PNG before it reaches the model. Images are cropped, not rescaled, so a larger value zero-pads and a smaller one throws away the object's edges. It is also the resolution the backbone is built at, which matters for ViT/Swin/inception. Match it to the crop size used when the dataset was generated. Default 224.
- **`init_weights`** — (bool) - Start the backbone from ImageNet-pretrained weights instead of random initialisation; the spaCR classifier head bolted on top is randomly initialised either way. Leave it on - transfer learning converges in far fewer epochs on the small annotated sets typical here. Turn it off only to train from scratch on a very large dataset, or to measure how much pretraining contributes. Default True.
- **`intermedeate_save`** — (bool) - Intended to control whether extra checkpoints are written mid-run when validation accuracy crosses 99, 98, 95 or 94 percent, on top of the final-epoch save. It currently has no effect: train_model passes that threshold list to the saver unconditionally, so those checkpoints are written regardless of this flag. Default True.
- **`label_smoothing`** — (float) - Epsilon passed to cross-entropy when loss_type is label_smoothing: each target keeps 1 - eps of its probability mass and the rest is spread across the other classes. Raise it (typically 0.05-0.2) when the model gets over-confident or annotations are noisy; 0 disables. Ignored by every other loss type. Default 0.1.
- **`leakage_audit_train_test`** — (bool) - Audit the permanent train/ and test/ boundary before any classifier fit. Checks plate/well/field/object lineage, exported augmentation families and (when enabled) byte-identical renamed copies. Default True. API: spacr.classifier_evaluation.audit_dataset_splits.
- **`leakage_hash_content`** — (bool) - SHA-256 hash classifier images during leakage audits so an identical crop copied or renamed across train/test or CV boundaries is still detected. Reads files in 1 MiB chunks and never decodes pixels. Default True. API: spacr.classifier_evaluation.audit_cv_folds.
- **`leakage_require_identity`** — (bool) - Treat filenames that do not encode the protected cv_group_by identity, and files that cannot be hashed, as a failed audit rather than an advisory warning. Default True because independence cannot be claimed when lineage is unknown. API: spacr.classifier_evaluation.audit_split_leakage.
- **`learning_rate`** — (float) - Step size passed to the optimizer. Too high and the loss spikes or flatlines at chance; too low and training crawls or settles in a poor minimum. 1e-3 suits training from scratch, while 1e-4 to 1e-5 is safer when fine-tuning ImageNet weights (init_weights=True). The chosen schedule decays this starting value over the run. Default 0.001.
- **`logit_adjust_tau`** — (float) - Strength of the Menon-et-al. logit adjustment: tau * log(class prior) is added to the logits during training, pulling decisions toward rare classes. Only used when loss_type resolves to logit_adjust_ce, which 'auto' picks when the smallest class is under 10% of the data. Higher tau corrects harder; 0 disables. Default 1.0.
- **`loss_type`** — (str) - Loss used to train the classifier. For a 2+ class head: 'focal_loss' (down-weights easy examples), 'cross_entropy', 'label_smoothing' (epsilon 0.1), 'ce_weighted' (inverse-frequency class weights), 'logit_adjust_ce' and 'asl'. 'binary_cross_entropy_with_logits' is legal ONLY for a single-logit head and raises otherwise. Reach for a weighted or focal loss when the classes are imbalanced, which for a screen they usually are. Default 'focal_loss'.
- **`max_failure_rate`** — (float or None) - Fraction of failed items above which the run aborts rather than finishing and reporting. 0.2 means 'stop once more than a fifth of the fields have failed', on the grounds that whatever is left is no longer the experiment. The ledger is stamped into the artifact before the abort, so the evidence survives. None (the default) never aborts on rate alone - every failure is still counted and reported, and the artifact is still marked partial. Default None.
- **`metadata_type_by`** — (str) - Which png_list column the class_metadata values are matched against when dataset_mode is 'metadata'. Normally 'columnID' or 'rowID' -- the two well-metadata columns filepaths_to_database writes -- but any png_list column is accepted, including 'condition' once annotate_conditions has added it. If the values in class_metadata do not appear in this column, the class simply gets no crops rather than erroring. Default 'columnID'.
- **`model_path`** — (str) - Path to a trained spaCR classifier saved as a whole PyTorch object (loaded with torch.load(weights_only=False), not a state_dict). Used when applying a model to a dataset tar and when generating activation maps. deep_spacr overwrites it with the freshly trained model whenever train is True, so set it only to score with an existing model. Default ''.
- **`model_type`** — (str) - Backbone architecture for the single-object image classifier: any TorchVision classification model name (resnet50, maxvit_t, densenet121, ...). An unrecognised name is NOT fatal when it is read -- choose_model prints 'Invalid model_type' and returns None, and training fails afterwards -- and the special name 'custom' passes the name check then raises NotImplementedError. Bigger backbones need more memory and more labelled crops to beat a smaller one. Default 'maxvit_t'.
- **`n_jobs`** — (int) - CPU workers for parallel stages: measurement, mask adjustment, DataLoader loading, and the sklearn/UMAP calls where -1 means every core. Raise it to shorten CPU-bound steps until RAM or disk I/O saturates. Note the measure-and-crop pipeline overrides your value with cpu_count()-4. Defaults vary by pipeline: cpu_count()-4, -1, or None.
- **`n_top_examples`** — (int) - Number of highest-confidence images saved per predicted class after full-dataset inference. This gives a quick visual check of class meaning and common errors. Default 20.
- **`nested_cv_inner_folds`** — (int) - Number of inner grouped folds used inside every outer CV fold. 0 (default) keeps the faster ordinary grouped CV; 2 or more trains one inner model per fold, uses inner validation for early stopping/model selection, ensembles those models, and evaluates only once on the untouched outer fold. Runtime is approximately outer_folds x inner_folds training runs, but the outer score is not reused for tuning. API: spacr.classifier_evaluation.nested_group_folds.
- **`normalization`** — (str) - Which normalisation the images get: 'imagenet' (the mean and standard deviation the pretrained backbones were trained with), 'dataset' (this dataset's own statistics), 'percentile' (per-image contrast stretch), 'none', or 'custom'. Default 'imagenet'.
- **`normalization_scope`** — (str) - Whether normalisation statistics are computed per 'image', per 'batch', or once over the whole 'dataset'. Per-image is the safest default: batch statistics leak information between the objects in a batch, and dataset statistics have to be recomputed whenever the dataset changes. Default 'image'.
- **`normalize`** — (bool) - Percentile-normalize each image channel (2nd to 98th percentile, clipped to 0-1) before display or model input; in the activation-map tool this rescales the image the CAM/saliency heatmap is drawn over. Turn it on when raw channels are too dim to read under the overlay. Affects display and input scaling only, never stored pixels. Default True.
- **`object_array`** — (str) - On-demand crops: which object the crops are cut around - 'cell', 'nucleus', 'pathogen', 'cytoplasm' or 'organelle'. Its mask plane in merged/*.npy is what defines each object's extent. Default 'cell'.
- **`optimizer_type`** — (str) - PyTorch optimizer used by deep_spacr.train_model: 'adamw', 'adam', 'adamax', 'sgd', 'rmsprop', 'nadam', 'radam', 'adagrad', 'adadelta' or 'asgd'. AdamW is the robust fine-tuning default; SGD can generalise better but usually needs more epochs. amsgrad applies only to Adam/AdamW. API: spacr.deep_spacr.train_model(optimizer_type=...). Default 'adamw'.
- **`path_string`** — (str) - A substring that must appear in a crop's path for it to join the dataset, e.g. 'cell_png' or 'nucleus_png'. It was called png_type, which named a type it never was: this is a path filter and nothing more. The old name still works. Default 'cell_png'.
- **`pin_memory`** — (bool) - Decode and hold the entire train/test image set in RAM up front (loaded in parallel across all cores) and hand batches to the GPU from page-locked memory. Enable when the dataset fits comfortably in RAM and disk I/O is the bottleneck; disable for large datasets or it will exhaust memory before the first epoch even starts. Default False.
- **`plot`** — (bool) - Render and save QC figures while the pipeline runs: channel montages and Cellpose mask overlays during segmentation, before/after filtration views and crop grids during measurement. It adds figures per batch, so a full plate becomes much slower and more memory-hungry; keep it for small or test_mode runs, which force it on. Default False.
- **`random_seed`** — (int) - Reproducibility seed shared by labelled train/test splitting, train/validation splitting, and grouped cross-validation folds. Keep it fixed to reproduce a run; change it to test sensitivity to one lucky split. Default 42.
- **`resume_checkpoint`** — (str) - Path to a spaCR training artifact to continue exactly: restores model, optimizer, scheduler, epoch, best score and random-generator state. Use custom_model_path instead when only the weights should be reused. Default ''.
- **`sample`** — (int, list or None) - Randomly draw this many PNG crops from the database when building the dataset tar instead of using all of them; a list uses its first element, and values above the total are clamped. Use it to build a quick trial dataset or to cap a huge screen. None uses every crop, shuffled. Default None.
- **`schedule`** — (str) - Learning-rate scheduler used by spacr.deep_spacr.train_model: 'cosine', 'cosine_warm_restarts', 'reduce_lr_on_plateau', 'step_lr', 'exponential', 'linear', or 'none'. Plateau reacts to validation loss; cosine and linear use the epoch budget; warm restarts periodically raise the rate to escape a narrow minimum. API: train_model(schedule=...). Default 'cosine'.
- **`score_threshold`** — (float) - Probability cutoff (0-1) applied to the model's positive-class score when deriving the binary cv_predictions column: pred &gt;= threshold becomes 1. The raw probability is always saved alongside it, so this only changes the hard call, not the score. Lower it to catch more positives at the cost of false positives; raise it for precision. Default 0.5.
- **`src`** — (str, path) - Folder the current step reads from and writes into: raw images for mask generation, the merged/ folder of .npy stacks for measure, the plate root for dataset/regression steps, or the folder of .fastq.gz reads for sequencing. Outputs (stack/, masks/, measurements/measurements.db, datasets/, results/) are created inside it. A list of paths, or a "['a','b']" string, processes several plates in one run. No usable default: the settings factories fill a placeholder ('path' or '/path/to/src'), so this must be supplied.
- **`strict_errors`** — (bool or None) - What happens when a step hits a problem it could survive. OFF records the failure in the run ledger and the end-of-run summary and carries on with the items that worked. ON raises immediately on a setup or configuration error -- an unreadable path, a missing column, a database that will not open -- so a batch stops at the first sign its inputs are wrong instead of producing a plausible partial result. Per-item failures such as one corrupt image are survived either way. None defers to $SPACR_STRICT_ERRORS, which is how a cluster sets it for a whole batch. Default None.
- **`tables`** — (list) - Measurement tables read from each plate's database and merged into one analysis frame. Only 'cell', 'nucleus', 'pathogen', 'cytoplasm' and 'png_list' are actually merged. Any other name -- INCLUDING 'organelle', which the measure step does write -- is loaded and then dropped, so asking for it costs time and returns nothing, with no warning that the table you wanted is missing from the result. Default ['cell', 'nucleus', 'pathogen', 'cytoplasm'].
- **`tar_path`** — (str) - Existing full-dataset tar to reuse for inference. Leave blank to generate one beneath the first plate's datasets folder. Multiple selected plates are combined into one tar. API: spacr.deep_spacr.apply_model_to_tar. Default empty.
- **`tensorboard`** — (bool) - Write live PyTorch loss, accuracy, macro-F1 and learning-rate events to dst/tensorboard while the vision model trains. Open that folder with tensorboard --logdir PATH for an interactive dashboard that can compare runs. The in-app zoomable loss/accuracy monitor is controlled separately by plot. Default True.
- **`test`** — (bool) - In classifier training, run the held-out evaluation pass (combine with train, or use alone to score an existing model). In the sequencing barcode mapper it means something different: process only the first read chunk and print a preview, so you can sanity-check the regex and barcode CSVs in seconds. Default False.
- **`test_split`** — (float) - Fraction of the generated crops held out as the test set, between 0 and 1. The split respects the grouping level chosen elsewhere, so crops from one well do not straddle it and the score is not inflated by the model recognising the well. Raising it buys a steadier estimate and costs training data. Default 0.1.
- **`train`** — (bool) - Run the training stage. Turn OFF to apply an existing model_path to a dataset without retraining, which is the usual way to score a new plate with a model trained earlier. Default True.
- **`train_channels`** — (list) - Which colour planes of each object crop the classifier sees, chosen from 'r', 'g' and 'b'. Fewer channels means a smaller input tensor and a model that cannot use the dropped stain, so drop a channel only when it carries no signal for your phenotype. The joined letters also become part of the saved model's filename. Default ['r', 'g', 'b'].
- **`use_checkpoint`** — (bool) - Run the backbone's forward pass through torch.utils.checkpoint: intermediate activations are discarded and recomputed during the backward pass, trading extra compute for a large drop in activation memory. Enable when a bigger batch_size or image_size gives CUDA out-of-memory; disable for the fastest epochs when VRAM is not the constraint. Default True.
- **`val_split`** — (float) - Fraction of src/train randomly held out as a validation set each run (0.1 = 10 percent). The validation score drives checkpoint selection, early stopping and the live training curves; at 0 there is no validation loader, so checkpointing falls back to training accuracy, which rewards memorisation. Raise it on small datasets for a less noisy estimate. With a grouping level set the holdout is whole groups, so the realised share is quantised to them and can land well off what you asked for; both numbers are reported rather than quietly rounded. Default 0.1.
- **`verbose`** — (bool) - Print extra run detail instead of the minimal log: the resolved settings table, the channel and model choices per object type, per-table row counts, and how many objects survive each filter. It only adds console output, so turn it on when object counts come out unexpected and you need to see which stage removed them. The default differs per pipeline -- True for mask, UMAP, screen analysis, barcode mapping and Cellpose training; False for measure, the plotting helpers and regression.
- **`weight_decay`** — (float) - L2 penalty applied to the weights on every optimizer step (AdamW applies it decoupled from the gradient). Raise it, toward 1e-3 to 1e-2, when validation loss climbs while training loss keeps falling; lower it toward 0 when the model cannot fit the training set at all. Every supported optimizer honours it. Default 0.00001.

## 4. Run

After editing the settings cell, run the function cell immediately below it. Long operations report progress through spaCR's logging system; set `SPACR_LOG_LEVEL=DEBUG` before starting Jupyter for more detail.

In [ ]:
settings = {
    'amsgrad': True,
    'annotation_column': 'test',
    'apply_model_to_dataset': True,
    'augment': False,
    'balance_to_smallest': True,
    'batch_size': 64,
    'channel_of_interest': 3,
    'class_balance': 'none',
    'class_folder_names': ['nc', 'pc'],
    'class_metadata': [['c1'], ['c2']],
    'classes': {},
    'classifier_evaluation': True,
    'coordinate_columns': None,
    'crop_shape': 'bounding_box',
    'crop_source': 'pre_generated',
    'cross_validation_enabled': False,
    'cross_validation_folds': 0,
    'custom_model': False,
    'custom_model_path': '',
    'cv_group_by': 'well',
    'dataset': '',
    'dataset_mode': 'metadata',
    'dropout_rate': 0.1,
    'early_stopping_patience': 0,
    'epochs': 100,
    'evaluation_bins': 10,
    'evaluation_calibration': 'temperature',
    'evaluation_fail_on_leakage': True,
    'experiment': 'exp.',
    'extract_channels': [0, 1, 2],
    'file_metadata': None,
    'file_type': 'cell_png',
    'focal_alpha': None,
    'focal_gamma': 2.0,
    'generate_full_dataset': False,
    'generate_training_dataset': True,
    'gradient_accumulation': True,
    'gradient_accumulation_steps': 4,
    'image_size': 224,
    'init_weights': True,
    'intermedeate_save': True,
    'label_smoothing': 0.1,
    'leakage_audit_train_test': True,
    'leakage_hash_content': True,
    'leakage_require_identity': True,
    'learning_rate': 0.001,
    'logit_adjust_tau': 1.0,
    'loss_type': 'auto',
    'max_failure_rate': None,
    'metadata_type_by': 'columnID',
    'model_path': '',
    'model_type': 'maxvit_t',
    'n_jobs': max(1, (__import__('os').cpu_count() or 1) - 4),
    'n_top_examples': 20,
    'nested_cv_inner_folds': 0,
    'normalization': 'imagenet',
    'normalization_scope': 'image',
    'normalize': True,
    'object_array': 'cell',
    'optimizer_type': 'adamw',
    'path_string': 'cell_png',
    'pin_memory': False,
    'plot': True,
    'random_seed': 42,
    'resume_checkpoint': '',
    'sample': None,
    'schedule': 'cosine',
    'score_threshold': 0.5,
    'src': 'path',
    'strict_errors': None,
    'tables': None,
    'tar_path': '',
    'tensorboard': True,
    'test': False,
    'test_split': 0.1,
    'train': True,
    'train_channels': ['r', 'g', 'b'],
    'use_checkpoint': True,
    'val_split': 0.1,
    'verbose': True,
    'weight_decay': 1e-05,
}

In [ ]:
deep_spacr(settings)

## Where the output went

A trained model, a held-out performance report, and per-object scores merged back into the measurement database.

spaCR writes beside the source folder rather than into a global location, so a plate stays self-contained and re-running does not clobber a different experiment.

### Next steps

* The GUI covers the same workflows with the settings laid out as a form — `python -m spacr`.
* The narrated walkthroughs are at <https://einarolafsson.github.io/spacr/tutorials/>.
* The API reference is at <https://einarolafsson.github.io/spacr/>.